## RCAN Training

A Residual Channel Attention Network (RCAN) is trained for image super-resolution using paired low-resolution (LR) and high-resolution (HR) images.

The model uses the RCAN architecture with 10 residual groups, 20 residual blocks per group, and 64 feature channels. Transfer learning is applied by loading compatible weights from a pretrained RCAN model, while newly introduced layers are initialized randomly.

Training uses L1 loss, Adam optimization, linear learning-rate warmup, cosine annealing, gradient clipping, and early stopping based on validation PSNR.

In [ ]:
import os
import glob
import random
import time
import numpy as np

from pathlib import Path
from PIL import Image
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from torch.utils.data import Dataset, DataLoader

from model.rcan import Net


# =====================================================
# CONFIGURATION
# =====================================================
SCALE = 2

NUM_GROUPS = 10
NUM_BLOCKS = 20
NUM_CHANNELS = 64
REDUCTION = 16
RES_SCALE = 1.0

BATCH_SIZE = 16
NUM_EPOCHS = 100

LR_WARMUP_START = 1e-5
LR_MAX = 1e-4
LR_MIN = 1e-6
WARMUP_EPOCHS = 5

GRADIENT_CLIP = 10.0
EARLY_STOP_PATIENCE = 30
NUM_FREEZE_GROUPS = 4

RANDOM_STATE = 42

TRAIN_LR_DIR = Path("path/to/train/LR_up")
TRAIN_HR_DIR = Path("path/to/train/HR")

VAL_LR_DIR = Path("path/to/val/LR_up")
VAL_HR_DIR = Path("path/to/val/HR")

PRETRAINED_PATH = Path("path/to/pretrained_model.pth")
CHECKPOINT_PATH = Path(f"path/to/checkpoint_rcan_x{SCALE}.pth")
BEST_MODEL_PATH = Path(f"path/to/best_model_rcan_x{SCALE}.pth")

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}


# =====================================================
# REPRODUCIBILITY
# =====================================================
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


# =====================================================
# DEVICE
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# =====================================================
# RCAN MODEL SETUP
# =====================================================
model_config = type(
    "RCANConfig",
    (),
    {
        "scale": SCALE,
        "num_groups": NUM_GROUPS,
        "num_blocks": NUM_BLOCKS,
        "num_channels": NUM_CHANNELS,
        "reduction": REDUCTION,
        "res_scale": RES_SCALE,
    },
)()

model = Net(model_config).to(device)


# =====================================================
# LOAD COMPATIBLE PRETRAINED WEIGHTS
# =====================================================
if PRETRAINED_PATH.exists():

    pretrained_state = torch.load(
        PRETRAINED_PATH,
        map_location=device
    )

    model_state = model.state_dict()

    compatible_weights = {
        key: value
        for key, value in pretrained_state.items()
        if key in model_state
        and model_state[key].shape == value.shape
    }

    model_state.update(compatible_weights)
    model.load_state_dict(model_state)

    print(
        f"Loaded {len(compatible_weights)} compatible "
        f"layers from pretrained model."
    )

else:
    print("Pretrained model not found. Training from scratch.")


# =====================================================
# DATASET
# =====================================================
class SRDataset(Dataset):

    def __init__(self, lr_dir, hr_dir, train=True):
        self.lr_paths = sorted(
            glob.glob(
                os.path.join(str(lr_dir), "*", "*")
            )
        )

        self.lr_paths = [
            path for path in self.lr_paths
            if Path(path).suffix.lower() in VALID_EXTENSIONS
        ]

        self.hr_paths = [
            path.replace(
                str(lr_dir),
                str(hr_dir)
            )
            for path in self.lr_paths
        ]

        self.train = train

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x * 255.0)
        ])

    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self, index):

        lr_image = Image.open(
            self.lr_paths[index]
        ).convert("RGB")

        hr_image = Image.open(
            self.hr_paths[index]
        ).convert("RGB")

        # Synchronized augmentation
        if self.train:

            if random.random() < 0.5:
                lr_image = TF.hflip(lr_image)
                hr_image = TF.hflip(hr_image)

            if random.random() < 0.5:
                lr_image = TF.vflip(lr_image)
                hr_image = TF.vflip(hr_image)

            if random.random() < 0.5:
                angle = random.choice([90, -90])

                lr_image = TF.rotate(
                    lr_image,
                    angle
                )

                hr_image = TF.rotate(
                    hr_image,
                    angle
                )

        lr_image = self.transform(lr_image)
        hr_image = self.transform(hr_image)

        return lr_image, hr_image


# =====================================================
# DATALOADER
# =====================================================
train_dataset = SRDataset(
    TRAIN_LR_DIR,
    TRAIN_HR_DIR,
    train=True
)

val_dataset = SRDataset(
    VAL_LR_DIR,
    VAL_HR_DIR,
    train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Training samples   : {len(train_dataset):,}")
print(f"Validation samples : {len(val_dataset):,}")


# =====================================================
# FREEZE RCAN GROUPS
# =====================================================
def freeze_blocks(model, num_freeze_groups):

    body = list(model.body)
    total_groups = len(body)

    for index, block in enumerate(body):

        if index < total_groups - 1:
            trainable = index >= num_freeze_groups
        else:
            trainable = True

        for parameter in block.parameters():
            parameter.requires_grad = trainable


freeze_blocks(
    model,
    NUM_FREEZE_GROUPS
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    f"Trainable parameters: "
    f"{trainable_parameters:,} / "
    f"{total_parameters:,}"
)


# =====================================================
# LOSS, OPTIMIZER & SCHEDULER
# =====================================================
criterion = nn.L1Loss()

optimizer = torch.optim.Adam(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=LR_WARMUP_START,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS - WARMUP_EPOCHS,
    eta_min=LR_MIN
)


# =====================================================
# TRAINING HISTORY
# =====================================================
train_losses = []
train_psnrs = []
train_ssims = []

val_losses = []
val_psnrs = []
val_ssims = []

best_val_psnr = 0.0
best_val_ssim = 0.0
early_stop_counter = 0
start_epoch = 0


# =====================================================
# LINEAR LEARNING-RATE WARMUP
# =====================================================
def get_warmup_lr(
    epoch,
    warmup_epochs,
    start_lr,
    max_lr
):

    return start_lr + (
        max_lr - start_lr
    ) * (epoch / warmup_epochs)


# =====================================================
# LOAD CHECKPOINT
# =====================================================
if CHECKPOINT_PATH.exists():

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1

    train_losses = checkpoint["train_losses"]
    train_psnrs = checkpoint["train_psnrs"]
    train_ssims = checkpoint["train_ssims"]

    val_losses = checkpoint["val_losses"]
    val_psnrs = checkpoint["val_psnrs"]
    val_ssims = checkpoint["val_ssims"]

    best_val_psnr = checkpoint.get(
        "best_val_psnr",
        0.0
    )

    best_val_ssim = checkpoint.get(
        "best_val_ssim",
        0.0
    )

    early_stop_counter = checkpoint.get(
        "early_stop_counter",
        0
    )

    print(
        f"Resuming training from epoch "
        f"{start_epoch}"
    )


# =====================================================
# TRAINING
# =====================================================
start_time = time.time()

for epoch in range(
    start_epoch,
    NUM_EPOCHS
):

    # -------------------------
    # Learning-rate schedule
    # -------------------------
    if epoch < WARMUP_EPOCHS:

        current_lr = get_warmup_lr(
            epoch,
            WARMUP_EPOCHS,
            LR_WARMUP_START,
            LR_MAX
        )

        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        phase = "Warmup"

    else:

        current_lr = optimizer.param_groups[0]["lr"]
        phase = "Cosine"


    # -------------------------
    # Training phase
    # -------------------------
    model.train()

    epoch_loss = 0.0
    psnr_total = 0.0
    ssim_total = 0.0
    sample_count = 0

    for lr_batch, hr_batch in tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [{phase}]"
    ):

        lr_batch = lr_batch.to(device)
        hr_batch = hr_batch.to(device)

        sr_batch = model(lr_batch)

        loss = criterion(
            sr_batch,
            hr_batch
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter:
                parameter.requires_grad,
                model.parameters()
            ),
            max_norm=GRADIENT_CLIP
        )

        optimizer.step()

        epoch_loss += loss.item()

        # -------------------------
        # Training metrics
        # -------------------------
        with torch.no_grad():

            sr_eval = torch.clamp(
                sr_batch,
                0.0,
                255.0
            )

            sr_numpy = sr_eval.cpu().numpy()
            hr_numpy = hr_batch.cpu().numpy()

            for index in range(
                lr_batch.size(0)
            ):

                sr_image = np.transpose(
                    sr_numpy[index],
                    (1, 2, 0)
                )

                hr_image = np.transpose(
                    hr_numpy[index],
                    (1, 2, 0)
                )

                psnr_total += psnr(
                    hr_image,
                    sr_image,
                    data_range=255.0
                )

                ssim_total += ssim(
                    hr_image,
                    sr_image,
                    channel_axis=2,
                    data_range=255.0
                )

                sample_count += 1


    # -------------------------
    # Training summary
    # -------------------------
    avg_train_loss = (
        epoch_loss /
        len(train_loader)
    )

    avg_train_psnr = (
        psnr_total /
        sample_count
    )

    avg_train_ssim = (
        ssim_total /
        sample_count
    )

    train_losses.append(
        avg_train_loss
    )

    train_psnrs.append(
        avg_train_psnr
    )

    train_ssims.append(
        avg_train_ssim
    )


    # -------------------------
    # Validation phase
    # -------------------------
    model.eval()

    val_loss_total = 0.0
    val_psnr_total = 0.0
    val_ssim_total = 0.0
    val_sample_count = 0

    with torch.no_grad():

        for val_lr, val_hr in val_loader:

            val_lr = val_lr.to(device)
            val_hr = val_hr.to(device)

            val_sr = model(val_lr)

            val_sr = torch.clamp(
                val_sr,
                0.0,
                255.0
            )

            val_loss_total += criterion(
                val_sr,
                val_hr
            ).item()

            val_sr_numpy = (
                val_sr.cpu().numpy()
            )

            val_hr_numpy = (
                val_hr.cpu().numpy()
            )

            for index in range(
                val_lr.size(0)
            ):

                sr_image = np.transpose(
                    val_sr_numpy[index],
                    (1, 2, 0)
                )

                hr_image = np.transpose(
                    val_hr_numpy[index],
                    (1, 2, 0)
                )

                val_psnr_total += psnr(
                    hr_image,
                    sr_image,
                    data_range=255.0
                )

                val_ssim_total += ssim(
                    hr_image,
                    sr_image,
                    channel_axis=2,
                    data_range=255.0
                )

                val_sample_count += 1


    # -------------------------
    # Validation summary
    # -------------------------
    avg_val_loss = (
        val_loss_total /
        len(val_loader)
    )

    avg_val_psnr = (
        val_psnr_total /
        val_sample_count
    )

    avg_val_ssim = (
        val_ssim_total /
        val_sample_count
    )

    val_losses.append(
        avg_val_loss
    )

    val_psnrs.append(
        avg_val_psnr
    )

    val_ssims.append(
        avg_val_ssim
    )


    # -------------------------
    # Print epoch results
    # -------------------------
    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train PSNR: {avg_train_psnr:.2f} | "
        f"Train SSIM: {avg_train_ssim:.4f}"
    )

    print(
        f"Validation Loss: {avg_val_loss:.4f} | "
        f"Validation PSNR: {avg_val_psnr:.2f} | "
        f"Validation SSIM: {avg_val_ssim:.4f}"
    )


    # -------------------------
    # Best model & early stopping
    # -------------------------
    if avg_val_psnr > best_val_psnr:

        best_val_psnr = avg_val_psnr
        best_val_ssim = avg_val_ssim

        early_stop_counter = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print(
            f"Best model saved | "
            f"PSNR: {best_val_psnr:.4f} | "
            f"SSIM: {best_val_ssim:.4f}"
        )

    else:

        early_stop_counter += 1

        print(
            f"No improvement | "
            f"Early stopping: "
            f"{early_stop_counter}/"
            f"{EARLY_STOP_PATIENCE}"
        )

        if (
            early_stop_counter
            >= EARLY_STOP_PATIENCE
        ):
            print(
                "Early stopping triggered."
            )
            break


    # -------------------------
    # Update scheduler
    # -------------------------
    if epoch >= WARMUP_EPOCHS:
        scheduler.step()


    # -------------------------
    # Save checkpoint
    # -------------------------
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_losses": train_losses,
            "train_psnrs": train_psnrs,
            "train_ssims": train_ssims,
            "val_losses": val_losses,
            "val_psnrs": val_psnrs,
            "val_ssims": val_ssims,
            "best_val_psnr": best_val_psnr,
            "best_val_ssim": best_val_ssim,
            "early_stop_counter": early_stop_counter
        },
        CHECKPOINT_PATH
    )


# =====================================================
# TRAINING SUMMARY
# =====================================================
elapsed_time = time.time() - start_time

print("\n" + "=" * 60)
print("Training completed")
print("=" * 60)
print(f"Scale          : ×{SCALE}")
print(f"Best Val PSNR  : {best_val_psnr:.4f}")
print(f"Best Val SSIM  : {best_val_ssim:.4f}")
print(f"Training time  : {elapsed_time / 3600:.2f} hours")